<a href="https://colab.research.google.com/github/w-olivine29/data-analysis/blob/main/%EC%98%A4%ED%94%88%EC%86%8C%EC%8A%A4_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EB%B6%84%EC%84%9D_14%EA%B0%95_%EC%8B%A4%EC%8A%B5_%EB%AF%B8%EC%99%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 14강 비정형 데이터 분석 : 패션 사진 데이터 활용

### 목표

- 비정형 데이터를 인공지능 모델로 분석하여 실무에서 활용 가능한 보고서 형태로 가공

- 패션 트렌드라는 구체적인 주제를 통해, 비정형 데이터 분석의 실질적인 활용 방안을 경험하고자 함


### 분석 프로세스 개요

1. 데이터 수집
  - requests를 이용한 RSS 데이터 수집
  - lxml을 이용한 XML 파싱
  - 이미지 데이터 추출
2. VLM을 이용한 이미지 분석
  - 프롬프트를 이용한 이미지 필터링
  - 프롬프트를 이용한 스타일 분석
3. LLM을 이용한 키워드 분석 및 보고서 작성
  - 텍스트 전처리
  - 색상 및 스타일 키워드 추출
  - 워드 클라우드 분석
  - 보고서 작성

# 주의 : 런타임 GPU 로 설정 필요

In [5]:
# 4bit VLM 처리를 위한 bitsandbytes 설치
# LLM 처리를 위한 VLLM 설치 (오래걸리는 작업(>5분)이므로 미리 실행!)
#!pip install bitsandbytes==0.45.3 vllm==0.7.3 transformers==4.48.2 (버전 문제로 설치 불가)


# pip 도구 먼저 최신화
!pip install --upgrade pip

# 버전을 지정하지 않고 현재 환경에 맞는 최신 호환 버전으로 설치
!pip install bitsandbytes vllm transformers

# 필요 시 세션 재시작

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 28.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 855.2/855.2 kB 26.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 40.9 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.20.3
    Uninstalling tokenizers-0.20.3:
      Successfully uninstalled tokenizers-0.20.3
  Attempting uninstall: transformers
    Found existing installation: transformers 4.46.3
    Uninstalling transformers-4.46.3:
      Successfully uninstalled transformers-4.46.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conf

In [1]:
# 한글 처리를 위한 matplotlib 설정 (1)

!sudo apt-get install -y fonts-nanum
!sudo fc-cache –fv
!rm ~/.cache/matplotlib -rf

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-nanum is already the newest version (20200506-1).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.


- 런타임 -> 세션 다시 시작

In [1]:
# 한글 처리를 위한 matplotlib 설정 (2)

import matplotlib.pyplot as plt
plt.rc('font', family='NanumBarunGothic')

# 1. 데이터 수집 및 전처리

## 14-1 RSS 피드에서 이미지 URL 추출

In [1]:
import requests
from lxml import etree
from lxml.html import fromstring
import pandas as pd

def extract_unique_images(rss_url):
    ## 주어진 RSS 피드 URL에서 고유한 이미지 URL들을 추출하는 함수 정의
    try:

        ## requests 라이브러리를 사용하여 RSS 피드 URL로부터 내용을 가져옴
        response = requests.get(rss_url)

        ## 가져온 XML 응답 내용을 lxml의 etree.fromstring으로 파싱하여 XML 트리 root를 생성
        root = etree.fromstring(response.content)

        image_urls = set()

        ## XML 트리에서 모든 'item' 태그를 XPath를 사용하여 순회
        for item in root.xpath('//item'):
            description = item.find('description')

            if description is not None and description.text:
                ## description의 텍스트 내용을 lxml.html.fromstring으로 파싱하여 HTML 트리를 생성
                html_tree = fromstring(description.text)

                ## HTML 트리에서 첫 번째 <img> 태그의 'src' 속성 값을 XPath를 사용하여 추출
                img_url = html_tree.xpath('string(//img/@src)')

                if img_url:
                    image_urls.add(img_url)

        return list(image_urls)

    except Exception as e:
        ## 오류 발생 시 오류 메시지를 출력하고 빈 리스트를 반환
        print(f"Error occurred: {e}")
        return []

rss_url = "https://glltn.com/feed/"
## extract_unique_images 함수를 호출하여 고유한 이미지 URL들을 추출
unique_images = extract_unique_images(rss_url)

## 추출된 이미지 URL 리스트를 사용하여 'image'라는 열을 가진 pandas DataFrame을 생성
df = pd.DataFrame(unique_images, columns=["image"])

df

,image
0,https://glltn.com/wp-content/blogs.dir/1/files...
1,https://glltn.com/wp-content/blogs.dir/1/files...
2,https://glltn.com/wp-content/blogs.dir/1/files...
3,https://glltn.com/wp-content/blogs.dir/1/files...
4,https://glltn.com/wp-content/blogs.dir/1/files...
5,https://glltn.com/wp-content/blogs.dir/1/files...
6,https://glltn.com/wp-content/blogs.dir/1/files...
7,https://glltn.com/wp-content/blogs.dir/1/files...
8,https://glltn.com/wp-content/blogs.dir/1/files...
9,https://glltn.com/wp-content/blogs.dir/1/files...


## 14-2 수집 데이터 확인

In [2]:
from IPython.display import display, HTML

def path_to_image_html(path):
    ## 이미지 경로를 HTML img 태그로 변환하는 함수
    return f'<img src="{path}" width="300" />'

## DataFrame의 스타일을 설정하여 이미지 너비를 300px로 지정
df.style.set_table_styles([{'selector': 'img', 'props': 'width: 300px;'}])

## DataFrame을 HTML로 변환하여 출력. 이미지 열은 path_to_image_html 함수로 포맷팅
display(HTML(df.to_html(escape=False, formatters=dict(**{'image': path_to_image_html}))))

,image
0,
1,
2,
3,
4,
5,
6,
7,
8,
9,


## 2. VLM을 이용한 이미지 분석

## 14-3 VLM 모델 로드

In [4]:
# [MiniCPM 모델 로드 에러 해결을 위한 다운그레이드]
# 앞선 셀에서 최신 버전으로 설치를 완료했으나, (제공된 코드의 버전은 에러로 인한 설치 실패로 최신 버전 다운)
# 최신 transformers 라이브러리가 MiniCPM 모델 코드와 충돌함(all_tied_weights_keys 에러).
# 파이썬 3.13에서 빌드 에러가 안 나면서 + 모델 충돌도 피할 수 있는 4.46.3 버전으로 덮어쓰기
!pip install transformers==4.46.3

# 후에 세션 다시 시작

In [5]:
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer

## 'openbmb/MiniCPM-V-2_6-int4' 모델을 사전 훈련된 가중치와 함께 로드
## trust_remote_code=True는 허브에서 사용자 정의 코드를 실행할 수 있도록 허용
model = AutoModel.from_pretrained('openbmb/MiniCPM-V-2_6-int4', trust_remote_code=True)


## 로드된 모델에 해당하는 토크나이저를 로드
tokenizer = AutoTokenizer.from_pretrained('openbmb/MiniCPM-V-2_6-int4', trust_remote_code=True)


## 모델을 평가 모드로 설정 (드롭아웃 등 훈련 시에만 필요한 기능 비활성화)
model.eval()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
`low_cpu_mem_usage` was None, now default to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

MiniCPMV(
  (llm): Qwen2ForCausalLM(
    (model): Qwen2Model(
      (embed_tokens): Embedding(151666, 3584)
      (layers): ModuleList(
        (0-27): 28 x Qwen2DecoderLayer(
          (self_attn): Qwen2SdpaAttention(
            (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
            (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
            (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
            (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
            (rotary_emb): Qwen2RotaryEmbedding()
          )
          (mlp): Qwen2MLP(
            (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
            (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
            (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
            (act_fn): SiLU()
          )
          (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
          (

## 14-4 이미지 질문 응답 예시

실습에 제공된 이미지로 실습이 불가하기에 다른 이미지로 대체하였음

![고양이 이미지](https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?fm=jpg&q=85&w=800)

In [30]:
# 이미지 요청 정상 여부 테스트 용

import requests

image_url = 'https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?fm=jpg&q=85&w=800'

response = requests.get(image_url, timeout=30)

print("상태 코드:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("응답 크기:", len(response.content))

상태 코드: 200
Content-Type: image/jpeg
응답 크기: 50620


In [31]:
from transformers import set_seed
from PIL import Image
from io import BytesIO
import requests

## 재현성을 위해 시드(seed)를 42로 설정
set_seed(42)


## 예시 이미지 URL 정의 (제시된 이미지 url로는 실습 진행이 불가하기에 다른 url로 세팅함)
image_url = 'https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?fm=jpg&q=85&w=800'



# requests로 이미지 다운로드 후 PIL Image 객체로 열고 RGB 형식으로 변환
# image = Image.open(requests.get(image_url, stream=True).raw).convert('RGB')


## requests로 이미지 다운로드 후 PIL Image 객체로 열고 RGB 형식으로 변환
response = requests.get(image_url, timeout=30)
response.raise_for_status()
image = Image.open(BytesIO(response.content)).convert('RGB')


# 이미지에 대한 질문 정의
question = 'how many cats in the photo?'


## 모델 입력 형식에 맞춰 메시지 구성 (이미지와 질문 포함)
msgs = [{'role': 'user', 'content': [image, question]}]


## 모델의 chat 함수를 호출하여 이미지와 질문에 대한 응답 생성
result = model.chat(image=None, msgs=msgs, tokenizer=tokenizer)


## 모델의 응답 출력
print(result)

1


In [33]:
set_seed(42)

## 이미지에 대한 질문을 업데이트
question = 'Describe the photo.'


## 모델 입력 형식에 맞춰 메시지 구성 (이전에 로드된 이미지와 업데이트된 질문 포함)
msgs = [{'role': 'user', 'content': [image, question]}]


## 모델의 chat 함수를 호출하여 업데이트된 질문에 대한 응답 생성
result = model.chat(image=None, msgs=msgs, tokenizer=tokenizer)


## 모델의 응답 출력
print(result)

The photo shows a black and white cat with striking green eyes, positioned in front of a green background. The cat is resting its paws on what appears to be a brown wooden surface, possibly a piece of furniture or a ledge. Its gaze is directed straight at the camera, giving an impression of curiosity or alertness. The lighting is soft, enhancing the contrast between the cat's fur colors and the solid color of the background. There are no other discernible objects or animals in the frame, placing the entire focus on the cat itself.


In [34]:
set_seed(42)
## 새로운 이미지에 대한 질문 (고양이의 품종, 외향적 특징 묻기)
question = 'What breed of cat is shown in the photo? Describe its appearance and explain whether its breed can be reliably identified from the image alone.'


## 모델 입력 형식에 맞춰 메시지 구성 (이전에 로드된 이미지와 설명 요청 질문 포함)
msgs = [{'role': 'user', 'content': [image, question]}]


## 모델의 chat 함수를 호출하여 이미지에 대한 설명을 생성
result = model.chat(image=None, msgs=msgs, tokenizer=tokenizer)


## 모델의 응답 (이미지 설명) 출력
print(result)

The cat in the photo appears to be a domestic shorthair, which is a common breed known for its mixed ancestry. It has a distinctive black and white coat with a tuxedo-like pattern. While it's challenging to determine the exact mix of breeds from an image alone without genetic testing or pedigree information, the physical characteristics suggest it could be part of this popular breed category. Domestic shorthairs come in various colors and patterns, so while we can identify it as such, pinpointing specific breeds within the domestic shorthair category would require more information.


## 14-5 의류 이미지 여부 판단

In [35]:
def is_picture_of_clothing(image_url):
    ## 이미지 URL이 의류 사진인지 판단하는 함수
    # 의류가 포함된 사진인지 확인하는 질문 작성 (영어로)
    question = 'Is this a picture of clothing? MUST say yes or no.'

    image = Image.open(requests.get(image_url, stream=True).raw).convert('RGB')
    msgs = [{'role': 'user', 'content': [image, question]}]

    result = model.chat(image=None, msgs=msgs, tokenizer=tokenizer, temperature=0.1)
    print(result)

    ## 응답에 'yes'가 포함되어 있는지 확인하여 True/False 반환
    return 'yes' in result.lower()

## DataFrame의 'image' 열에 함수를 적용하여 'is_clothing' 열에 결과 저장
df['is_clothing'] = df['image'].apply(is_picture_of_clothing)

Yes
Yes.
Yes, this image is a picture of clothing. It showcases a man's outfit which includes several distinct pieces such as a coat, trousers, scarf, and hat. The focus on the attire suggests that it may be used for fashion-related purposes, possibly to display or advertise the garments.
Yes.
Yes, this image is of clothing. It showcases a coordinated outfit consisting of a jacket and shorts with a distinctive pattern of black crosses on a cream-colored fabric. The attire appears to be designed for casual wear, possibly targeting a youthful demographic given the playful nature of the cross motifs.
No.
No, the image is not of clothing. It appears to be a book cover with an artistic design that includes a photograph and text. The presence of a house in the photograph suggests it could relate to themes of home or rural life, but there are no visible elements that indicate it is related to clothing.
No.
No, the image is not of clothing. It appears to be a photograph from a book or publicat

## 14-6 의류 판단 결과 시각화

In [36]:
display(HTML(df.to_html(escape=False, formatters=dict(**{'image': path_to_image_html}))))

,image,is_clothing
0,,True
1,,True
2,,True
3,,True
4,,True
5,,False
6,,False
7,,False
8,,False
9,,True


## 14-7 의류 이미지 필터링

In [37]:
## 'is_clothing' 열의 값이 True인 행들만 필터링하여 DataFrame을 업데이트
df = df[df['is_clothing']]

## 14-8 의류 스타일 분석

In [38]:
def describe_style(image_url):
    ## 주어진 이미지 URL의 의류 스타일을 분석하는 함수
    question = 'Analyze the style of the clothes. Please let me explain the colors and trend changes.'

    image = Image.open(requests.get(image_url, stream=True).raw).convert('RGB')
    msgs = [{'role': 'user', 'content': [image, question]}]


    ## 모델의 chat 함수를 호출하여 이미지에 대한 스타일 분석 응답 생성
    result = model.chat(image=None, msgs=msgs, tokenizer=tokenizer)
    return result

## 필터링된 DataFrame의 'image' 열에 describe_style 함수를 적용
## 결과는 'style'이라는 새로운 열에 저장
df['style'] = df['image'].apply(describe_style)

/tmp/ipykernel_11405/2036135887.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['style'] = df['image'].apply(describe_style)


In [39]:
display(HTML(df.to_html(escape=False, formatters=dict(**{'image': path_to_image_html}))))

,image,is_clothing,style
0,,True,"The boots in the image reflect a style that is both functional and fashionable, likely appealing to those who favor classic designs with a touch of vintage flair. The color palette consists of deep browns, which are versatile and timeless, suggesting these boots could be styled for various occasions from casual outings to more formal events.\n\nIn terms of trend changes, such boots have seen a resurgence in popularity due to their durability and comfort, as well as their ability to complement a wide range of outfits. The use of leather and suede materials indicates an appreciation for quality craftsmanship, while the addition of fringe detailing adds a modern twist to the otherwise traditional design. This blend of classic elements with contemporary accents suggests a trend towards retro-inspired fashion that values both heritage and innovation.\n\nOverall, the boots represent a shift towards embracing practical yet stylish footwear that can transition seamlessly between different styles and settings, reflecting a broader trend in fashion where functionality meets aesthetic appeal."
1,,True,"The style of the shoes in the image is indicative of a casual, comfortable aesthetic often associated with moccasin or slip-on footwear. These types of shoes are typically designed for ease of wear and indoor use, making them popular choices for lounging at home or light outdoor activities where comfort is prioritized over formality.\n\nIn terms of color, the black hue suggests versatility and a timeless appeal, as it can easily match with various outfits and does not go out of style. The presence of fur lining adds to the warmth and coziness, which may indicate that these shoes are intended for colder weather or seasons when additional insulation is desired.\n\nRegarding trend changes, while the basic design of moccasins has remained consistent over time, variations in materials, colors, and added features like elastic side panels (as seen on one shoe) have become more prevalent. This evolution reflects a blend of traditional craftsmanship with modern convenience and fashion trends."
2,,True,"The style of the clothes in the image leans towards a vintage-inspired aesthetic, possibly drawing from mid-20th-century fashion. The overcoat is reminiscent of trench coats popularized in the 1930s and 40s, characterized by its long length, wide lapels, and functional design elements like toggle closures.\n\nThe use of muted colors such as grey and beige suggests a preference for understated elegance rather than bold statements. These color choices are timeless and versatile, often associated with classic styles that have stood the test of time. The distressed look on the coat adds a modern twist to the otherwise traditional piece, indicating a trend towards incorporating worn textures into contemporary fashion.\n\nThe hat, likely a fedora or flat cap, complements the overall vintage theme, being another staple accessory from past decades. Its dark color provides a stark contrast to the lighter tones of the coat and trousers, adding depth to the outfit.\n\nUnderneath the coat, the cream-colored sweater paired with a textured scarf introduces a layering technique that is both practical and stylish. Layering has been a significant trend in recent years, allowing for flexibility in adapting outfits to different weather conditions while also creating visual interest through varied textures and patterns.\n\nIn summary, the clothing style reflects a blend of vintage inspiration with modern touches, emphasizing comfort, functionality, and an appreciation for classic designs updated with contemporary details."
3,,True,"The style of the clothes in the image reflects a blend of casual and streetwear fashion trends. The leopard print lining on the brown jacket is indicative of a trend that often incorporates animal prints, which have been popular in various fashion eras but are particularly revived in modern streetwear. This c

# 3. LLM을 이용한 키워드 분석 및 보고서 작성

## 14-9 언어 모델(LLM) 로드

In [43]:
import sys
import importlib.metadata as md

print("Python:", sys.version)

for package in ["vllm", "transformers", "torch", "tokenizers"]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: 설치되지 않음")


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
vllm: 0.26.0
transformers: 4.46.3
torch: 2.11.0+cu130
tokenizers: 0.20.3


In [44]:
from importlib.metadata import metadata

requirements = metadata("vllm").get_all("Requires-Dist") or []

for req in requirements:
    if "transformers" in req.lower() or "tokenizers" in req.lower():
        print(req)


transformers>=5.5.3
tokenizers>=0.21.1


In [45]:
%pip install -U "transformers>=5.5.3" "tokenizers>=0.21.1"

  Using cached transformers-5.21.1-py3-none-any.whl.metadata (31 kB)
  Using cached tokenizers-0.23.3-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.8 kB)
  Using cached huggingface_hub-2.3.0-py3-none-any.whl.metadata (16 kB)
Using cached transformers-5.21.1-py3-none-any.whl (13.0 MB)
Using cached tokenizers-0.23.3-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.4 MB)
Using cached huggingface_hub-2.3.0-py3-none-any.whl (855 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.20.3
    Uninstalling tokenizers-0.20.3:
      Successfully uninstalled tokenizers-0.20.3
  Attempting uninstall: transformers
    Found existing installation: transformers 4.46.3
    Uninstalling transformers-4.46.3:
      Successfully uninstalled transformer

In [1]:
import transformers
import tokenizers
import vllm

print(transformers.__version__)
print(tokenizers.__version__)
print(vllm.__version__)

5.21.1
0.23.3
0.26.0


In [4]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"

In [ ]:
from vllm import LLM, SamplingParams

## vLLM 라이브러리를 사용하여 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct' 모델을 로드
## gpu_memory_utilization은 GPU 메모리 사용 비율을 0.5로 설정
## max_model_len은 모델이 처리할 수 있는 최대 토큰 길이를 10000으로 설정

llm = LLM(model='LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct',gpu_memory_utilization=0.5, max_model_len=10000, trust_remote_code=True)


## 14-10 색상 정보 추출

In [ ]:
from vllm import SamplingParams ## SamplingParams 임포트가 필요

def extract_color(style):
  ## 주어진 스타일 설명 텍스트에서 색상을 한글로 추출하는 함수
  prompt = [
      {
          "role": "system",
          "content": "You are EXAONE model from LG AI Research, a helpful assistant."
      },
      {
          "role": "user",
          "content": f"다음의 글에서 색상을 한글로 추출해주세요. 색상 외에 다른 정보는 적지 말아주세요.\n{style}" # vlm이 작성한 글에서 색상 정보 추출, 한글로 번역하면서
      }
  ]
  ## 샘플링 파라미터 설정 (온도, top_p, 최대 토큰 수)
  sampling_params = SamplingParams(temperature=0.2, top_p=0.95, max_tokens=1024)
  ## LLM 모델을 사용하여 프롬프트에 대한 응답 생성
  result = llm.chat(prompt, sampling_params)[0].outputs[0].text
  print(result)
  return result

## DataFrame의 'style' 열에 extract_color 함수를 적용
## 결과는 'color'라는 새로운 열에 저장
df['color'] = df['style'].apply(extract_color)

## 14-11 스타일 키워드 추출

In [ ]:
from vllm import SamplingParams ## SamplingParams 임포트가 필요

def extract_color(style):
  ## 주어진 스타일 설명 텍스트에서 스타일 키워드를 한글로 추출하는 함수
  prompt = [
      {
          "role": "system",
          "content": "You are EXAONE model from LG AI Research, a helpful assistant."
      },
      {
          "role": "user",
          "content": f"다음의 글에서 스타일 키워드를 한글로 추출해주세요. 스타일 키워드 외에 다른 정보는 적지 말아주세요.\n{style}" # vlm이 작성한 글에서 스타일 키워드 추출, 한글로 번역하면서
      }
  ]
  ## 샘플링 파라미터 설정 (온도, top_p, 최대 토큰 수)
  sampling_params = SamplingParams(temperature=0.2, top_p=0.95, max_tokens=1024)
  ## LLM 모델을 사용하여 프롬프트에 대한 응답 생성
  result = llm.chat(prompt, sampling_params)[0].outputs[0].text
  print(result)
  return result

## DataFrame의 'style' 열에 extract_color 함수를 적용 (함수 이름은 이전과 동일하지만 기능 변경)
## 결과는 'keyword'라는 새로운 열에 저장
df['keyword'] = df['style'].apply(extract_color)

In [ ]:
display(HTML(df.to_html(escape=False, formatters=dict(**{'image': path_to_image_html}))))

## 14-12 텍스트 데이터 정제

In [ ]:
import re

def clean_text(text):
    ## 텍스트에서 특수 문자 및 HTML 태그를 제거하고 소문자로 변환하는 함수
    if isinstance(text, str):
       ## 영문, 숫자, 한글, 공백을 제외한 모든 문자 제거
       text = re.sub(r'[^a-zA-Z0-9가-힣\s]', '', text)
       ## HTML 태그 제거
       text = re.sub(r'<[^>]*>', '', text)
       ## 텍스트를 소문자로 변환
       text = text.lower()
       return text
    else:
        return ""

## 'color' 열의 텍스트 데이터 정제
df['color'] = df['color'].apply(clean_text)
## 'keyword' 열의 텍스트 데이터 정제
df['keyword'] = df['keyword'].apply(clean_text)

## 14-13 워드 클라우드 생성 및 시각화

In [ ]:
from collections import Counter
from wordcloud import WordCloud
import matplotlib.pyplot as plt

def get_word_count(df):
    ## DataFrame의 'color'와 'keyword' 열에서 단어 빈도를 계산하는 함수
    if not df.empty:
        ## 'color' 열의 모든 단어를 리스트로 합침
        all_nouns = df['color'].apply(str.split).sum()
        ## 'keyword' 열의 모든 단어를 추가
        all_nouns += df['keyword'].apply(str.split).sum()
        ## '색상' 단어를 제외한 모든 단어를 필터링
        all_nouns = [word for word in all_nouns if word not in ['색상']]
        ## 단어 빈도를 Counter 객체로 반환
        return Counter(all_nouns)
    return Counter() ## DataFrame이 비어있으면 빈 Counter 반환

def create_wordcloud(word_count):
    ## 단어 빈도수를 기반으로 워드 클라우드를 생성하고 시각화하는 함수
    if not word_count: ## 단어 빈도가 없으면 워드클라우드 생성하지 않음
        print("No words to generate word cloud.")
        return

    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color='white',
        colormap='viridis',
        font_path='/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf' ## 한글 폰트 경로 지정
        ).generate_from_frequencies(word_count)

    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off") ## 축 표시 제거
    plt.show() ## 워드 클라우드 출력

## DataFrame에서 단어 빈도 계산
word_count = get_word_count(df)
## 계산된 단어 빈도로 워드 클라우드 생성 및 시각화
create_wordcloud(word_count)

## 14-14 트렌드 분석 보고서 생성 프롬프트 구성 및 실행

## 14-15 분석 보고서 시각화

In [ ]:
from vllm import SamplingParams ## SamplingParams 임포트가 필요

## 시스템 메시지로 시작하는 프롬프트 리스트 초기화
prompt = [
    {
        "role": "system",
        "content": "You are EXAONE model from LG AI Research, a helpful assistant."
    }
]
## DataFrame의 각 행을 순회하며 '스타일 노트'와 '이미지 URL'을 사용자 메시지로 추가
for row in df.itertuples():
  prompt.append({"role": "user", "content": f'스타일 노트: {row.style}\n이미지 url:{row.image}'})
## 마지막으로, 종합적인 트렌드 분석 보고서 작성을 요청하는 사용자 메시지 추가
## 보고서 제목, 내용의 전문성, 마크다운 형식, 예시 이미지 포함을 지시
prompt.append({"role": "user", "content": '주어진 스타일 노트를 토대로 종합적인 트렌드 방향의 분석 보고서를 작성해주세요. 보고서의 제목은 해외 룩북 스타일 분석입니다. 내용은 전문적이면서 명확하게 작성해주세요. 문서 형식은 마크다운 입니다. 예시 이미지를 적절한 곳에 추가해서 바로 볼 수 있도록 해주세요.'})

## 샘플링 파라미터 설정 (온도, top_p, 최대 토큰 수)
sampling_params = SamplingParams(temperature=0.2, top_p=0.95, max_tokens=4096)
## LLM 모델을 사용하여 구성된 프롬프트에 대한 응답 생성
result = llm.chat(prompt, sampling_params)[0].outputs[0].text

In [ ]:
from IPython.display import display, Markdown

## LLM으로부터 생성된 결과(Markdown 형식의 보고서)를 Jupyter 환경에 표시
display(Markdown(result))